In [5]:
!mamba list torch
!mamba list transformers
!mamba list datasets
!mamba list trl
!mamba list accelerate

List of packages in environment: "C:\\Users\\uqpua\\miniforge3"

  Name           Version  Build                       Channel    
-------------------------------------------------------------------
  libtorch       2.5.1    cpu_mkl_hf54a72f_117        conda-forge
  pytorch        2.5.1    cpu_mkl_py312_h9ecdb75_117  conda-forge
  pytorch-mutex  1.0      cpu                         pytorch    
  torchaudio     2.5.1    py312_cpu                   pytorch    
  torchvision    0.20.1   cpu_py312_he303efe_6        conda-forge
List of packages in environment: "C:\\Users\\uqpua\\miniforge3"

  Name          Version  Build         Channel    
----------------------------------------------------
  transformers  4.57.6   pyhd8ed1ab_0  conda-forge
List of packages in environment: "C:\\Users\\uqpua\\miniforge3"

  Name      Version  Build         Channel    
------------------------------------------------
  datasets  4.0.0    pyhcf101f3_0  conda-forge
List of packages in environment: "C:\\Users

In [6]:
# !mamba install -y transformers datasets
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                        TrainingArguments, Trainer)

C:\Users\uqpua\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
train_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
ds = load_dataset("trl-lib/ultrafeedback_binarized")

Generating test split: 100%|█████████████████████████████████████████████| 1000/1000 [00:00<00:00, 47250.71 examples/s]


In [8]:
train_dataset, ds

(Dataset({
     features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
     num_rows: 62135
 }),
 DatasetDict({
     train: Dataset({
         features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
         num_rows: 62135
     })
     test: Dataset({
         features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
         num_rows: 1000
     })
 }))

In [17]:
# =========================================
# 1) 재현성(랜덤 시드) 고정 함수
# =========================================
def set_seed(seed: int = 42):
    random.seed(seed)          # 파이썬 랜덤 고정
    np.random.seed(seed)       # numpy 랜덤 고정
    torch.manual_seed(seed)    # torch 랜덤 고정


# =========================================
# 2) IMDB를 SFT용 (prompt/response)로 변환
# =========================================
def build_imdb_sft_splits(train_size: int = 100, val_size: int = 100):
    """
    IMDB 데이터셋에서 일부만 뽑아서
    prompt/response 형태의 데이터셋으로 바꿔 반환합니다.

    CPU-only라서 train_size/val_size를 작게 잡습니다.
    """

    # (1) IMDB 데이터셋 로드 (인터넷 다운로드)
    ds = load_dataset("imdb")  # ds["train"], ds["test"]

    # (2) CPU 학습 속도를 위해 일부만 선택
    train_raw = ds["train"].shuffle(seed=42).select(range(train_size))
    val_raw = ds["test"].shuffle(seed=42).select(range(val_size))

    # (3) 각 샘플을 prompt/response 형태로 변환
    def convert(example):
        # example["text"] : 영화 리뷰 텍스트
        # example["label"]: 0(negative), 1(positive)
        text = example["text"]
        label = example["label"]

        # 정답 라벨을 사람이 읽는 단어로 바꿈
        sentiment = "positive" if label == 1 else "negative"

        # prompt: 모델에게 '감성을 맞혀봐'라고 시키는 지시문
        # 너무 길면 토큰이 길어져 느려지므로, 리뷰를 일부만 자르는 것도 방법
        prompt = (
            "Instruction: Determine the sentiment of the movie review.\n"
            f"Review: {text}\n"
            "Answer (positive/negative):"
        )

        # response: 모델이 출력해야 할 정답 (앞에 공백 1개 넣으면 토큰화가 자연스러운 경우가 많음)
        response = f" {sentiment}"

        return {"prompt": prompt, "response": response}

    train_sft = train_raw.map(convert, remove_columns=train_raw.column_names)
    val_sft = val_raw.map(convert, remove_columns=val_raw.column_names)

    return train_sft, val_sft


# =========================================
# 3) 토크나이징 + 라벨 마스킹(-100) 적용
# =========================================
def tokenize_sft_dataset(train_sft, val_sft, tokenizer, max_length: int = 160):
    """
    SFT에서 핵심:
    - input_ids: prompt + response를 합친 토큰
    - labels: input_ids 복사한 뒤, prompt 구간은 -100으로 바꿔 loss에서 제외
    """

    # distilgpt2/gpt2는 pad_token이 없는 경우가 많아 padding할 때 에러가 날 수 있음
    # 그래서 eos_token을 pad_token으로 설정하는 게 CPU 시험에서 가장 안전합니다.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_and_mask(batch):
        prompts = batch["prompt"]
        responses = batch["response"]

        input_ids_list = []
        attention_mask_list = []
        labels_list = []

        for p, r in zip(prompts, responses):
            # (1) prompt만 토큰화해서 "prompt 토큰 길이"를 구함
            p_tok = tokenizer(p, add_special_tokens=False)

            # (2) prompt + response 전체를 토큰화해서 모델 입력을 만듦
            full_text = p + r
            full = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                add_special_tokens=False
            )

            input_ids = full["input_ids"]                 # 길이 max_length
            attention_mask = full["attention_mask"]       # padding은 0, 실제 토큰은 1

            # (3) labels는 input_ids를 그대로 복사 (기본은 모든 토큰을 맞히게 하는 것)
            labels = input_ids.copy()

            # (4) prompt 토큰 부분은 -100으로 바꿔서 loss에서 제외
            prompt_len = len(p_tok["input_ids"])
            prompt_len = min(prompt_len, max_length)  # 혹시 prompt가 너무 길면 방어

            for i in range(prompt_len):
                labels[i] = -100

            # (5) padding 토큰도 loss에서 제외(-100)
            # attention_mask가 0인 위치는 padding이라고 보면 됨
            for i in range(max_length):
                if attention_mask[i] == 0:
                    labels[i] = -100

            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            labels_list.append(labels)

        return {
            "input_ids": input_ids_list,
            "attention_mask": attention_mask_list,
            "labels": labels_list,
        }

    # map으로 전체 데이터셋 변환
    train_tok = train_sft.map(tokenize_and_mask, batched=True, remove_columns=train_sft.column_names)
    val_tok = val_sft.map(tokenize_and_mask, batched=True, remove_columns=val_sft.column_names)

    # Trainer가 torch 텐서로 받도록 지정
    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return train_tok, val_tok


# =========================================
# 4) 학습 실행 함수 (CPU-only)
# =========================================
def train_cpu_sft_imdb(
    model_name: str = "distilgpt2",
    train_size: int = 100,
    val_size: int = 100,
    max_length: int = 160,
    output_dir: str = "./result"
):
    """
    CPU-only 환경에서 돌아가도록:
    - 작은 모델(distilgpt2)
    - 작은 데이터(train_size/val_size)
    - 짧은 max_length
    - batch 작게
    - epoch 1
    """

    set_seed(42)
    print("CUDA available? ->", torch.cuda.is_available())  # 시험에서는 False가 정상

    # (1) 토크나이저/모델 로드
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # padding 안전장치

    model = AutoModelForCausalLM.from_pretrained(model_name)

    # (2) 데이터셋 준비 (IMDB → SFT 포맷)
    train_sft, val_sft = build_imdb_sft_splits(train_size=train_size, val_size=val_size)

    # (3) 토크나이징 + 마스킹
    train_ds, val_ds = tokenize_sft_dataset(train_sft, val_sft, tokenizer, max_length=max_length)

    # (4) Trainer 설정
    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",   # epoch 끝날 때 평가
        save_strategy="no",            # 저장은 보통 제출에서 불필요 + 용량 문제도 있음
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=5e-5,            # 작은 LM SFT에서 흔히 쓰는 값
        per_device_train_batch_size=2, # CPU에서 현실적인 값 (느리면 1로)
        per_device_eval_batch_size=2,
        num_train_epochs=1,            # CPU라 1 epoch 권장
        report_to="none",
        remove_unused_columns=False    # labels 유지 위해 매우 중요!
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer
    )

    # (5) 학습
    trainer.train()

    # (6) 평가 (eval_loss 출력)
    metrics = trainer.evaluate()
    print("Eval metrics:", metrics)

    # (7) 간단 생성 테스트 (학습이 되었는지 확인)
    test_review = "The plot was boring and the acting was terrible."
    prompt = (
        "Instruction: Determine the sentiment of the movie review.\n"
        f"Review: {test_review}\n"
        "Answer (positive/negative):"
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=4,   # " negative" 정도만 나오면 됨
            do_sample=False
        )

    print("\n=== Generated example ===")
    print(tokenizer.decode(out[0], skip_special_tokens=True))

    return metrics

In [18]:
# =========================================
# 5) 실행
# =========================================
# CPU가 너무 느리면 train_size=300, max_length=128로 줄이세요.
train_cpu_sft_imdb()

CUDA available? -> False


Map: 100%|███████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 455.13 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.448500,nan


Eval metrics: {'eval_loss': nan, 'eval_runtime': 36.5246, 'eval_samples_per_second': 2.738, 'eval_steps_per_second': 1.369, 'epoch': 1.0}

=== Generated example ===
Instruction: Determine the sentiment of the movie review.
Review: The plot was boring and the acting was terrible.
Answer (positive/negative): positive positive positive positive


{'eval_loss': nan,
 'eval_runtime': 36.5246,
 'eval_samples_per_second': 2.738,
 'eval_steps_per_second': 1.369,
 'epoch': 1.0}

In [21]:
# ============================================================
# Dolly 15k 기반 CPU-only SFT (완성 코드)
# - 환경: CPU only (GPU 없음)
# - 모델: distilgpt2 (가벼운 GPT2 변형)
# - 데이터: databricks/databricks-dolly-15k (instruction/response 구조)
# - 핵심: prompt 토큰은 labels=-100으로 마스킹하여 loss 계산에서 제외
# ============================================================

# (필요 시) 설치 확인/설치
# 에러가 나면 아래 주석을 풀고 설치해줘.
# !mamba install -y transformers datasets

import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)

# -----------------------------
# 1) 재현성(랜덤) 고정
# -----------------------------
def set_seed(seed: int = 42):
    """실행할 때마다 결과가 덜 흔들리도록 랜덤 시드를 고정"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# -----------------------------
# 2) Dolly 데이터셋 로드 + train/val 분리
# -----------------------------
def load_dolly_splits(train_size: int = 100, val_size: int = 100, seed: int = 42):
    """
    Dolly 15k는 기본적으로 'train' 하나만 제공되는 경우가 많아서
    우리가 직접 train/val로 나눠서 사용한다.

    CPU-only라서 train_size/val_size를 작게 잡는다.
    """
    ds = load_dataset("databricks/databricks-dolly-15k")["train"]

    # 섞은 뒤 일부만 사용 (CPU 속도/시간 때문에)
    ds = ds.shuffle(seed=seed)

    # 앞부분을 val로, 그 다음을 train으로 나눔 (간단한 방식)
    val_ds = ds.select(range(val_size))
    train_ds = ds.select(range(val_size, val_size + train_size))

    return train_ds, val_ds

# -----------------------------
# 3) Dolly 샘플을 prompt/response로 구성
# -----------------------------
def format_prompt_and_response(example):
    """
    Dolly 레코드는 보통 아래 컬럼을 가진다:
      - instruction (필수)
      - context (없거나 빈 문자열일 수 있음)
      - response (정답/assistant 답변)
      - category (있을 수 있음)

    우리는 SFT 학습을 위해:
      prompt: instruction(+context) + "Response:" 까지
      response: 실제 응답 텍스트
    로 분리한다.
    """
    instruction = example.get("instruction", "")
    context = example.get("context", "") or ""  # None이면 "" 처리
    response = example.get("response", "")

    # prompt 템플릿: 아주 흔한 instruction-tuning 형태
    # (중요) prompt 끝에 "Response:" 또는 "Answer:"를 넣어주면
    # 모델이 그 뒤를 "완성"하도록 학습시키기 좋다.
    if context.strip():
        prompt = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Context:\n"
            f"{context}\n\n"
            "### Response:\n"
        )
    else:
        prompt = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
        )

    # response는 모델이 생성해야 하는 정답
    # 앞에 공백/개행은 데이터에 따라 달라질 수 있는데,
    # 여기서는 prompt가 이미 '\n'로 끝나기 때문에 response는 그대로 둔다.
    return {"prompt": prompt, "response": response}

# -----------------------------
# 4) 토크나이징 + 마스킹(-100) 적용
# -----------------------------
def tokenize_and_mask_sft(train_ds, val_ds, tokenizer, max_length: int = 256):
    """
    SFT의 핵심:
      - input_ids = (prompt + response) 토큰
      - labels = input_ids 복사 후, prompt 토큰 위치는 -100 처리
        => loss는 response 토큰에서만 계산됨
    """

    # GPT2 계열(distilgpt2 포함)은 pad_token이 없는 경우가 많음
    # padding을 쓰려면 pad_token을 지정해야 에러가 안 남
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # (1) 먼저 prompt/response 컬럼 생성
    train_fmt = train_ds.map(format_prompt_and_response, remove_columns=train_ds.column_names)
    val_fmt = val_ds.map(format_prompt_and_response, remove_columns=val_ds.column_names)

    def tokenize_fn(batch):
        """
        batch는 여러 샘플을 한 번에 받는다(batched=True).
        batch["prompt"], batch["response"]는 리스트 형태.
        """
        prompts = batch["prompt"]
        responses = batch["response"]

        input_ids_list = []
        attention_mask_list = []
        labels_list = []

        for p, r in zip(prompts, responses):
            # 1) prompt만 토큰화 -> prompt 길이(토큰 개수) 구하기
            p_tok = tokenizer(p, add_special_tokens=False)

            # 2) prompt + response 전체를 토큰화 -> 모델 입력
            full_text = p + r
            full = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                add_special_tokens=False
            )

            input_ids = full["input_ids"]
            attention_mask = full["attention_mask"]

            # 3) labels는 input_ids 복사
            labels = input_ids.copy()

            # 4) prompt 부분은 -100으로 마스킹 (loss 계산 제외)
            prompt_len = len(p_tok["input_ids"])
            prompt_len = min(prompt_len, max_length)  # 혹시 prompt가 너무 길면 방어

            for i in range(prompt_len):
                labels[i] = -100

            # 5) padding 토큰도 loss 계산 제외
            # attention_mask가 0이면 padding 위치라고 보면 됨
            for i in range(max_length):
                if attention_mask[i] == 0:
                    labels[i] = -100

            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            labels_list.append(labels)

        return {
            "input_ids": input_ids_list,
            "attention_mask": attention_mask_list,
            "labels": labels_list
        }

    # (2) 실제 토큰화 + 마스킹 수행
    train_tok = train_fmt.map(tokenize_fn, batched=True, remove_columns=train_fmt.column_names)
    val_tok = val_fmt.map(tokenize_fn, batched=True, remove_columns=val_fmt.column_names)

    # (3) Trainer가 torch 텐서를 받도록 포맷 지정
    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return train_tok, val_tok

# -----------------------------
# 5) 학습 함수 (Trainer 사용)
# -----------------------------
def train_dolly_cpu_sft(
    model_name: str = "distilgpt2",
    train_size: int = 100,
    val_size: int = 100,
    max_length: int = 256,
    output_dir: str = "./dolly_sft_ckpt",
):
    """
    CPU-only 환경에서 돌아가도록 파라미터를 보수적으로 설정.
    - train_size/val_size: 작게
    - batch_size: 1~2
    - epoch: 1
    - save_strategy: no (제출 20MB 제한도 있고 보통 불필요)
    """

    set_seed(42)

    # 시험 환경은 GPU 없음
    print("CUDA available? ->", torch.cuda.is_available())

    # (1) tokenizer + model 로드
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)

    # (2) Dolly 데이터 로드 + 분리
    train_raw, val_raw = load_dolly_splits(train_size=train_size, val_size=val_size, seed=42)

    # (3) 토큰화 + 마스킹(-100)
    train_ds, val_ds = tokenize_and_mask_sft(train_raw, val_raw, tokenizer, max_length=max_length)

    # (4) Trainer 설정
    # CPU-only이므로 batch는 작게, epoch도 1로 권장
    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",     # epoch 끝날 때 평가
        save_strategy="no",              # 저장 안 함(시간/용량 절약)
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=5e-5,              # 작은 LM SFT에서 무난
        per_device_train_batch_size=2,   # CPU가 느리면 1로 낮추기
        per_device_eval_batch_size=2,
        num_train_epochs=1,
        report_to="none",
        remove_unused_columns=False,     # labels 같은 컬럼 유지에 중요!
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer
    )

    # (5) 학습
    trainer.train()

    # (6) 평가 (eval_loss 확인)
    metrics = trainer.evaluate()
    print("Eval metrics:", metrics)

    # (7) 생성 테스트: prompt만 주고 답이 이어서 생성되는지 확인
    test_instruction = "Write a short motivational message for someone learning programming."
    prompt = (
        "### Instruction:\n"
        f"{test_instruction}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,   # CPU라 너무 길게 하지 않기
            do_sample=False
        )

    print("\n=== Generated example ===")
    print(tokenizer.decode(out[0], skip_special_tokens=True))

    return metrics

# -----------------------------
# 6) 실행
# -----------------------------
# CPU가 너무 느리면 아래처럼 줄여서 돌려도 됨:
# - train_size=300
# - val_size=100
# - max_length=192 또는 128
# - per_device_train_batch_size=1 (위 args에서 수정)

# 실행 예시:
# train_dolly_cpu_sft()


In [22]:
train_dolly_cpu_sft()

CUDA available? -> False


Map: 100%|███████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 488.14 examples/s]
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19392\2385679537.py:235: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,3.146600,2.805173


Eval metrics: {'eval_loss': 2.805172920227051, 'eval_runtime': 58.4128, 'eval_samples_per_second': 1.712, 'eval_steps_per_second': 0.856, 'epoch': 1.0}

=== Generated example ===
### Instruction:
Write a short motivational message for someone learning programming.

### Response:
Write a short motivational message for someone learning programming.

Write a short motivational message for someone learning programming.

Write a short motivational message for someone learning programming.

Write a short motivational message for someone learning programming.

Write a short motivational message for someone learning programming.

Write a short motivational message for someone learning programming.

Write a short motivational message for someone learning


{'eval_loss': 2.805172920227051,
 'eval_runtime': 58.4128,
 'eval_samples_per_second': 1.712,
 'eval_steps_per_second': 0.856,
 'epoch': 1.0}

In [ ]:
# ============================================================
# Dolly 15k 기반 CPU-only SFT (TRL의 SFTTrainer 사용)
# - 데이터: databricks/databricks-dolly-15k
# - 모델: distilgpt2 (CPU에서도 비교적 가벼움)
# - 포맷: prompt/completion (TRL이 공식 지원)
# - 핵심: prompt-completion 포맷이면 completion만 loss로 계산 가능
# ============================================================

# ----------------------------
# 0) (필요 시) 패키지 설치
# ----------------------------
# HackerRank 환경에 trl이 없을 수 있어요. 에러 나면 아래 실행:
# !pip -q install trl
# 또는 (환경에 따라) !mamba install -y trl -c conda-forge
#
# transformers/datasets는 보통 이미 있거나 설치 가능:
# !mamba install -y transformers datasets

import random
import numpy as np
import torch

from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM

# TRL의 SFTTrainer / SFTConfig
from trl import SFTTrainer, SFTConfig


# ----------------------------
# 1) 시드 고정(재현성)
# ----------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ----------------------------
# 2) Dolly를 prompt/completion 포맷으로 변환
# ----------------------------
def dolly_to_prompt_completion(example):
    """
    Dolly 레코드(보통):
      - instruction
      - context (없거나 빈 문자열일 수 있음)
      - response
      - category (있을 수도)

    TRL SFTTrainer가 지원하는 prompt-completion 포맷:
      {"prompt": "...", "completion": "..."}  :contentReference[oaicite:2]{index=2}
    """
    instruction = example.get("instruction", "")
    context = example.get("context", "") or ""
    response = example.get("response", "")

    # prompt: 정답 이전까지 (모델에게 "여기서부터 답해" 라고 표시)
    if context.strip():
        prompt = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Context:\n"
            f"{context}\n\n"
            "### Response:\n"
        )
    else:
        prompt = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
        )

    # completion: 모델이 생성해야 할 정답 부분
    # 앞에 공백/개행을 넣을지는 취향인데,
    # prompt가 이미 '\n' 포함이므로 response를 그대로 둬도 OK
    completion = response

    return {"prompt": prompt, "completion": completion}


def load_dolly_train_val(train_size=800, val_size=200, seed=42):
    """
    Dolly 15k는 종종 split이 train 하나로만 제공되므로,
    shuffle 후 앞부분을 val로, 다음을 train으로 나눔.
    """
    ds = load_dataset("databricks/databricks-dolly-15k")["train"]

    ds = ds.shuffle(seed=seed)

    val_raw = ds.select(range(val_size))
    train_raw = ds.select(range(val_size, val_size + train_size))

    # prompt/completion으로 변환 + 원래 컬럼 삭제
    val_pc = val_raw.map(dolly_to_prompt_completion, remove_columns=val_raw.column_names)
    train_pc = train_raw.map(dolly_to_prompt_completion, remove_columns=train_raw.column_names)

    return train_pc, val_pc


# ----------------------------
# 3) 학습 함수 (SFTTrainer)
# ----------------------------
def train_dolly_sfttrainer_cpu(
    model_name="distilgpt2",
    train_size=800,
    val_size=200,
    max_length=256,
    output_dir="./dolly_sfttrainer_ckpt",
):
    """
    CPU-only 기준으로 안전한 세팅:
    - 작은 데이터
    - 짧은 max_length
    - 작은 batch
    - epoch 1
    """

    set_seed(42)
    print("CUDA available? ->", torch.cuda.is_available())  # 시험 환경이면 False가 정상

    # (1) tokenizer / model 로드
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # GPT2계열은 pad_token이 없어서 padding 시 에러가 날 수 있어요.
    # TRL/Trainer 내부에서 padding이 필요하므로 eos를 pad로 지정해 두는 게 안전합니다.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)

    # (2) 데이터 준비 (prompt/completion)
    train_ds, val_ds = load_dolly_train_val(train_size=train_size, val_size=val_size)

    # (3) SFTConfig (TRL 전용 TrainingArguments 비슷한 것)
    # 문서에 SFTConfig 파라미터들이 정리되어 있음. :contentReference[oaicite:3]{index=3}
    # - dataset_text_field: LM(text) 데이터일 때 사용
    # - prompt/completion 데이터는 컬럼이 prompt/completion이면 TRL이 인식
    # - completion_only_loss: prompt-completion일 때 completion만 loss로 계산하도록 강제 가능 :contentReference[oaicite:4]{index=4}
    sft_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=2,     # CPU면 1~2 추천
        per_device_eval_batch_size=2,
        learning_rate=5e-5,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="no",                # 제출/용량/시간 때문에 저장은 보통 생략
        report_to="none",
        max_length=max_length,             # 토큰 길이(길수록 느림)
        packing=False,                     # packing은 효율 좋지만 초심자/코테에선 OFF가 안전
        completion_only_loss=True,         # completion만 loss로 계산(명시적으로 안전) :contentReference[oaicite:5]{index=5}
        fp16=False,                        # CPU라 의미 없음
        bf16=False,
    )

    # (4) SFTTrainer 생성
    trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
    )

    # (5) 학습
    trainer.train()

    # (6) 평가 (eval_loss 확인)
    metrics = trainer.evaluate()
    print("Eval metrics:", metrics)

    # (7) 간단 생성 테스트 (학습 감각 확인)
    test_prompt = (
        "### Instruction:\n"
        "Write a short motivational message for someone learning programming.\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(test_prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    print("\n=== Generated example ===")
    print(tokenizer.decode(out[0], skip_special_tokens=True))

    return metrics


# ----------------------------
# 4) 실행
# ----------------------------
# CPU가 너무 느리면 아래처럼 줄이면 됩니다:
# train_dolly_sfttrainer_cpu(train_size=300, val_size=100, max_length=128)

# 기본 실행:
# train_dolly_sfttrainer_cpu()
